In [ ]:
from src.compare import compare_models, print_comparison

configs = [
    {"name": "base", "time_varying": False, "covariates": None},
    {"name": "+covariates", "time_varying": False,
     "covariates": ["rest_advantage", "temp_std", "wind_std"]},
    {"name": "+time_varying", "time_varying": True, "covariates": None},
]

comparison = compare_models(season, configs, train_window=8, samples=500, nsims=500)
print_comparison(comparison)

# NFL Hierarchical Bayesian Model

Fits a hierarchical Bayesian model to NFL game data to estimate team-level
attack and defense strengths. Uses these estimates to simulate future games
and generate spread predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from src.data import load_game_data
from src.model import bhm, simulate_team_seasons, predictions, plot_hdis

## Load Data

In [ ]:
season = 2024

# Load regular season games (first 14 weeks as training data)
train_weeks = list(range(1, 15))
df_train, teams = load_game_data(season, weeks=train_weeks)

print(f"Training on {len(df_train)} games from weeks {min(train_weeks)}-{max(train_weeks)}")
print(f"{len(teams)} teams")
df_train.head()

## Fit Model

In [ ]:
# Fit hierarchical Bayesian model
idata = bhm(df_train, metric='score', K=1, samples=1000)

# Check convergence
az.plot_trace(idata, var_names=['home', 'intercept', 'sd_att', 'sd_def'])
plt.tight_layout()
plt.show()

## Team Strength Estimates

In [ ]:
# Extract posterior means for attack and defense
atts_mean = idata.posterior['atts'].mean(dim=['chain', 'draw']).values
defs_mean = idata.posterior['defs'].mean(dim=['chain', 'draw']).values

team_strengths = teams.copy()
team_strengths['attack'] = atts_mean
team_strengths['defense'] = defs_mean
team_strengths = team_strengths.sort_values('attack', ascending=False)

print("Team Strength Estimates (higher attack = stronger offense, lower defense = stronger defense):")
print(team_strengths[['team', 'attack', 'defense']].to_string(index=False))

## Simulate and Predict

In [ ]:
# Load test weeks
test_weeks = list(range(15, 19))
df_test, _ = load_game_data(season, weeks=test_weeks)

# Simulate games
nsims = 1000
simuls = simulate_team_seasons(df_test, idata, nsims=nsims, metric='score')

# Generate predictions with 90% credible intervals
hdis = predictions(df_test, simuls, teams, nsims=nsims)

# Plot
fig, ax = plot_hdis(hdis, metric='score')
plt.show()

## Spread Predictions

In [ ]:
from src.backtest import predict_week, print_summary

# Predict test weeks using the fitted model
preds = predict_week(idata, df_test, nsims=500)

# Show per-game predictions
print(preds[['week', 'home_team', 'away_team', 'home_win_prob',
             'pred_spread', 'actual_spread', 'correct']].to_string(index=False))

# Summary
print_summary(preds, f"Base model (weeks {min(test_weeks)}-{max(test_weeks)})")

## Rolling Backtest

Train on 8-week windows, predict the next week, roll forward across the season.
Compare model variants to see what helps.